# Dual-Layer Pipeline Monitoring with LangSmith and Evidently

This notebook demonstrates the dual-layer fraud detection pipeline and its
monitoring infrastructure. The architecture combines a primary ML scorer
(XGBoost/gradient-boosted trees) with a complementary RAG+LLM layer for
gray-zone and high-value transactions. Two observability platforms provide
coverage across both layers:

- **LangSmith** -- real-time LLM trace observability, capturing latency,
  token usage, retrieval quality, and custom drift feedback scores for
  every RAG+LLM invocation.
- **Evidently AI** -- periodic statistical drift analysis producing
  standalone HTML reports for both the ML model's input features
  (covariate shift) and the RAG layer's embedding space (embedding drift).

We walk through the full pipeline lifecycle using the Sparkov synthetic
fraud dataset: scoring transactions, routing decisions, detecting drift
in both layers, and generating compound drift assessments.

In [ ]:
import sys
import os
import warnings

# Ensure the project root is on the Python path
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.loader import SparkovDataLoader
from src.visualization.plots import (
    plot_ml_score_distribution,
    plot_routing_breakdown,
    plot_dual_layer_drift_correlation,
)
from src.visualization.schematics import (
    draw_pipeline_architecture,
    draw_dual_layer_interaction,
)
from src.fraud_detection.ml_scorer import SimulatedMLScorer
from src.fraud_detection.pipeline import PipelineConfig
from src.monitoring.evidently_reporter import EvidentlyDriftReporter

# Load the Sparkov dataset
loader = SparkovDataLoader()
transactions_df = loader.load()
print(f"Loaded {len(transactions_df)} transactions from Sparkov dataset")
transactions_df.head()

## Pipeline Architecture

The dual-layer system routes every transaction through the primary ML
scorer first. Only transactions that fall into the gray zone
(configurable score thresholds) or exceed the high-value amount
threshold are escalated to the complementary RAG+LLM layer. This
design keeps latency low for the majority of clear-cut decisions while
reserving deeper analysis for ambiguous cases.

In [ ]:
# Visualize the dual-layer pipeline architecture
draw_pipeline_architecture()

## ML Model Scoring

The primary layer uses a `SimulatedMLScorer` that mimics the behaviour
of a trained XGBoost model. It produces a fraud probability in [0, 1]
based on heuristic rules applied to the enriched transaction features.
We score every transaction in the Sparkov dataset and examine the
resulting score distribution.

In [ ]:
from src.fraud_detection.transaction_processor import TransactionProcessor

scorer = SimulatedMLScorer()
processor = TransactionProcessor()

# Score each transaction
ml_scores = []
for _, row in transactions_df.iterrows():
    txn_dict = row.to_dict()
    try:
        txn = processor.validate(txn_dict)
        enriched = processor.enrich(txn)
        result = scorer.predict(enriched)
        ml_scores.append(result.score)
    except Exception:
        ml_scores.append(np.nan)

transactions_df["ml_score"] = ml_scores
valid_scores = transactions_df["ml_score"].dropna()
print(f"Scored {len(valid_scores)} transactions")
print(f"Score statistics: mean={valid_scores.mean():.4f}, "
      f"std={valid_scores.std():.4f}, "
      f"median={valid_scores.median():.4f}")

# Plot the ML score distribution
plot_ml_score_distribution(valid_scores.values)

## Decision Routing Analysis

The pipeline config defines three decision zones:

- **ML-only approve** -- score < `gray_zone_lower` (default 0.3)
- **ML-only decline** -- score > `gray_zone_upper` (default 0.7)
- **ML + LLM** -- score in [0.3, 0.7] (gray zone) or amount > $10,000

We classify all scored transactions into these tiers and visualize the
routing breakdown.

In [ ]:
config = PipelineConfig()

# Classify each transaction into its routing tier
def classify_tier(row):
    score = row["ml_score"]
    if pd.isna(score):
        return "unscored"
    # High-value transactions always go to ML+LLM
    amount = row.get("amount", 0)
    if amount > config.high_value_threshold:
        return "ml_plus_llm"
    # Gray zone check
    if config.gray_zone_lower <= score <= config.gray_zone_upper:
        return "ml_plus_llm"
    return "ml_only"

transactions_df["tier"] = transactions_df.apply(classify_tier, axis=1)

tier_counts = transactions_df["tier"].value_counts()
print("Routing breakdown:")
for tier, count in tier_counts.items():
    pct = 100.0 * count / len(transactions_df)
    print(f"  {tier}: {count} ({pct:.1f}%)")

# Plot routing breakdown as a pie chart
plot_routing_breakdown(tier_counts.to_dict())

## Dual-Layer Drift Interaction

When monitoring a dual-layer system, four distinct drift scenarios
can occur, each with different risk profiles and remediation strategies:

1. **No drift in either layer** -- nominal operations, no action needed.
2. **Feature drift only (ML layer)** -- the primary scorer is operating
   on shifted inputs but the RAG safety net remains intact.
3. **Embedding drift only (RAG layer)** -- the ML scorer continues to
   function normally but LLM analysis quality degrades for gray-zone
   transactions.
4. **Compound drift (both layers)** -- critical scenario requiring
   immediate intervention: the primary scorer is unreliable and the
   safety net is simultaneously degraded.

In [ ]:
# Draw the four drift interaction scenarios
draw_dual_layer_interaction()

In [ ]:
# Simulate embedding drift and feature drift over time,
# then plot their correlation on a dual-axis chart.

np.random.seed(42)
n_windows = 20
time_windows = [f"Week {i+1}" for i in range(n_windows)]

# Simulate embedding drift scores: gradual increase then stabilization
embedding_drift_scores = np.clip(
    np.cumsum(np.random.normal(0.005, 0.01, n_windows)), 0, 0.4
)

# Simulate feature drift scores: correlated but with some lag
feature_drift_scores = np.clip(
    np.cumsum(np.random.normal(0.003, 0.008, n_windows)), 0, 0.35
)

print("Simulated drift progression:")
print(f"  Embedding drift range: [{embedding_drift_scores.min():.4f}, "
      f"{embedding_drift_scores.max():.4f}]")
print(f"  Feature drift range:   [{feature_drift_scores.min():.4f}, "
      f"{feature_drift_scores.max():.4f}]")
print(f"  Correlation: {np.corrcoef(embedding_drift_scores, feature_drift_scores)[0,1]:.4f}")

# Plot dual-axis correlation chart
plot_dual_layer_drift_correlation(
    time_labels=time_windows,
    embedding_drift=embedding_drift_scores,
    feature_drift=feature_drift_scores,
)

## Evidently AI Drift Report

Evidently AI provides deep statistical drill-downs on feature and
embedding drift. Below we generate two reports:

1. **Feature drift report** -- compares the ML model's input feature
   distributions between a reference period and a simulated production
   period. Uses the KS test for numerical features and chi-squared
   for categorical features.
2. **Embedding drift report** -- compares reference and production
   embedding distributions using Evidently's model-based MMD metric.

In [ ]:
# Generate feature drift report using Evidently
reporter = EvidentlyDriftReporter(
    reports_dir="reports/evidently",
    embedding_dim=64,
)

# Split Sparkov data into reference and production periods
n_total = len(transactions_df)
split_idx = n_total // 2

numerical_features = ["amount", "hour_of_day", "day_of_week"]
categorical_features = ["merchant_category_code", "channel"]

# Build feature DataFrames from the available columns
feature_cols = [c for c in numerical_features + categorical_features if c in transactions_df.columns]
reference_features = transactions_df.iloc[:split_idx][feature_cols].copy()
production_features = transactions_df.iloc[split_idx:][feature_cols].copy()

print(f"Reference period: {len(reference_features)} transactions")
print(f"Production period: {len(production_features)} transactions")

# Generate the feature drift report
feature_summary = reporter.generate_feature_drift_report(
    reference_features=reference_features,
    production_features=production_features,
    numerical_features=[c for c in numerical_features if c in feature_cols],
    categorical_features=[c for c in categorical_features if c in feature_cols],
    save_html=False,
    report_name="nb04_feature_drift",
)

print(f"\nFeature Drift Summary:")
print(f"  Overall drift detected: {feature_summary.overall_drift_detected}")
print(f"  Share of drifted features: {feature_summary.share_drifted:.2%}")
print(f"  Drifted features: {feature_summary.drifted_features}")
print(f"  Per-feature drift scores:")
for feat, score in feature_summary.feature_drift_scores.items():
    print(f"    {feat}: {score:.4f}")

In [ ]:
# Generate embedding drift report using Evidently
# Use real sentence-transformer embeddings from the Sparkov dataset.
# Falls back to random vectors if sentence-transformers is not installed.

n_ref = 500
n_prod = 500

ref_slice = transactions_df.iloc[:n_ref]
prod_slice = transactions_df.iloc[split_idx : split_idx + n_prod]

ref_texts = SparkovDataLoader.generate_transaction_texts(ref_slice).tolist()
prod_texts = SparkovDataLoader.generate_transaction_texts(prod_slice).tolist()

try:
    from src.embeddings.generator import LocalEmbeddingGenerator

    generator = LocalEmbeddingGenerator()
    embedding_dim = generator.dimensions
    print(f"Using LocalEmbeddingGenerator (model: all-MiniLM-L6-v2, dim={embedding_dim})")

    reference_embeddings = generator.encode_texts(ref_texts)
    production_embeddings = generator.encode_texts(prod_texts)
except ImportError:
    print("WARNING: sentence-transformers is not installed. Falling back to random embeddings.")
    embedding_dim = 384
    np.random.seed(123)
    reference_embeddings = np.random.randn(n_ref, embedding_dim).astype(np.float32)
    production_embeddings = np.random.randn(n_prod, embedding_dim).astype(np.float32)

# Update reporter with the actual embedding dimension
reporter = EvidentlyDriftReporter(
    reports_dir="reports/evidently",
    embedding_dim=embedding_dim,
)

embedding_summary = reporter.generate_embedding_drift_report(
    reference_embeddings=reference_embeddings,
    production_embeddings=production_embeddings,
    save_html=False,
    report_name="nb04_embedding_drift",
)

print("Embedding Drift Summary:")
print(f"  Embedding drift detected: {embedding_summary.embedding_drift_detected}")
print(f"  Embedding drift score: {embedding_summary.embedding_drift_score:.4f}")
print(f"  Dataset drift detected: {embedding_summary.dataset_drift_detected}")
print(f"  Share of drifted components: {embedding_summary.share_of_drifted_columns:.2%}")
print(f"  Number of drifted columns: {embedding_summary.number_of_drifted_columns} / "
      f"{embedding_summary.number_of_columns}")

## Compound Drift Assessment

The `generate_dual_layer_report` method runs both the embedding drift
and feature drift analyses, then produces a compound risk assessment
that accounts for the interaction between the two layers. This is the
key integration point where Evidently's statistical depth is combined
with the pipeline's architectural awareness.

In [ ]:
# Run compound dual-layer drift assessment
dual_report = reporter.generate_dual_layer_report(
    reference_embeddings=reference_embeddings,
    production_embeddings=production_embeddings,
    reference_features=reference_features,
    production_features=production_features,
    numerical_features=[c for c in numerical_features if c in feature_cols],
    categorical_features=[c for c in categorical_features if c in feature_cols],
    save_html=False,
)

print("Dual-Layer Drift Report")
print("=" * 60)
print(f"Compound drift detected: {dual_report['compound_drift_detected']}")
print()
print("Risk Assessment:")
print(dual_report["risk_assessment"])
print()

emb = dual_report["embedding_summary"]
feat = dual_report["feature_summary"]
print(f"Embedding layer -- drift detected: {emb.embedding_drift_detected}, "
      f"score: {emb.embedding_drift_score:.4f}")
print(f"Feature layer   -- drift detected: {feat.overall_drift_detected}, "
      f"share drifted: {feat.share_drifted:.2%}")

## Key Takeaways

**When LangSmith adds the most value:**

- Real-time monitoring of every RAG+LLM invocation in the complementary
  layer, capturing latency spikes, token usage anomalies, and retrieval
  quality degradation as they happen.
- Per-trace drill-down for debugging individual gray-zone transaction
  assessments, including the full prompt, retrieved context, and LLM
  response.
- Custom feedback scores that feed back into the drift detection loop,
  enabling rapid detection of LLM quality regressions.
- Continuous streaming observability suitable for high-throughput
  production environments.

**When Evidently AI adds the most value:**

- Periodic (hourly/daily) statistical drift reports with rigorous
  hypothesis testing (KS test, chi-squared, MMD) that quantify the
  magnitude and significance of distributional shifts.
- Standalone HTML reports that can be shared with stakeholders, archived
  for compliance, or integrated into CI/CD drift gates.
- Compound drift assessment across both pipeline layers, identifying
  critical scenarios where the ML scorer and RAG safety net degrade
  simultaneously.
- Per-feature drill-down that pinpoints which specific input features
  are driving the overall drift signal.

**The combination is stronger than either tool alone:** LangSmith
catches real-time quality issues in the LLM layer, while Evidently
provides the statistical rigour needed for drift-based retraining
decisions and compliance reporting. Together, they ensure that both
the fast ML path and the deep LLM path remain reliable over time.